In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
CREATE WIDGET TEXT storageName default="storageproject99"

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

## EXTERNAL LOCATIONS

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_project`)
COMMENT 'Ubicación externa para las tablas delta del Data deltalake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_project`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_project`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_project`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential_project`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

## CATALOGO

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_dev
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para almacenar las tablas metastore;'

## ESQUEMAS

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_dev.raw;
CREATE SCHEMA IF NOT EXISTS catalog_dev.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_dev.silver;
CREATE SCHEMA IF NOT EXISTS catalog_dev.golden;

## TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.matricula (
ACAD_CAREER STRING,
STRM STRING,	
MODALIDAD STRING,
EMPLID STRING,
EMPLID_NAME STRING,
ACAD_PROG STRING,
ACAD_PLAN STRING,
COURSE STRING,
COURSE_DESC STRING,
PROM INTEGER,
TURN STRING,
TURN_DESC STRING,
CAMPUS STRING,
CYCLE STRING,
NBR_CLASS STRING,
SECTION STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/matricula"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.ciclo (
INSTITUTION STRING,
ACAD_CAREER	STRING,
STRM STRING,
DESCR	STRING,
DESCRSHORT STRING,
TERM_BEGIN_DT TIMESTAMP,
TERM_END_DT TIMESTAMP,
WEEKS_OF_INSTRUCT INTEGER,
TERM_CATEGORY STRING,
ACAD_YEAR INTEGER,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/ciclo"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.bronze.programa (
INSTITUTION STRING,
EFFDT STRING,
EFF_STATUS STRING,
ACAD_CAREER STRING,
ACAD_PROG	STRING,
DESCR	STRING ,
DESCRSHORT STRING, 
ACAD_GROUP STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/programa"

## TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.silver.matriculados_unicos_escuela (
ACAD_CAREER STRING,
STRM STRING,
STRM_DESCR STRING,
EMPLID STRING,
EMPLID_NAME STRING,
PROG_DESCR STRING,
ACAD_GROUP STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/matriculados_unicos_escuela"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.silver.programa_estatus_calif (
ACAD_CAREER STRING,
STRM STRING,
STRM_DESCR STRING,
EMPLID STRING,
EMPLID_NAME STRING,
ACAD_PROG STRING,
PROG_DESCR STRING,
NBR_CLASS STRING,
COURSE STRING,
COURSE_DESC STRING,
CALIFICATION INTEGER,
ESTADO STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/programa_estatus_calif"

## TABLAS GOLDEN

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.cant_matriculados_unicos_escuela_salud (
GRADO_ACADEMICO STRING,
CICLO_LECTIVO STRING,
CICLO_LECTIVO_DESCR STRING,
ALUMNO_ID STRING,
ALUMNO_NOMBRE STRING,
PROGRAMA STRING,
ESCUELA STRING
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/cant_matriculados_unicos_escuela_salud"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_dev.golden.programa_por_cant_estatus_calif (
GRADO_ACADEMICO STRING,
CICLO_LECTIVO STRING,
CICLO_LECTIVO_DESCR STRING,
ALUMNO_ID STRING,
ALUMNO_NOMBRE STRING,
PROGRAMA_ID STRING,
PROGRAMA STRING,
CLASE_NUMERO STRING,
CURSO_ID STRING,
CURSO_DESCR STRING,
CALIFICACION INTEGER,
CALIFICACION_ESTATUS STRING
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/programa_por_cant_estatus_calif"
